One row per financial statement, unified metric + derived ratios + red-flag booleans.

Source: 
* `bronze_financial_data`
* `silver_companies`


In [0]:
CATALOG = "credit_risk_project"
BRONZE_SCHEMA = f"{CATALOG}.credit_risk_project_bronze"
SILVER_SCHEMA = f"{CATALOG}.credit_risk_project_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

In [0]:
from pyspark.sql import functions as F

df_financial = spark.table(f"{BRONZE_SCHEMA}.bronze_financial_data")

# display(df_financial)
df_bilans = (
    df_financial
    .filter(F.col("Status") == "ok")
    .select(
        F.col("SIREN").cast("string").alias("siren"),
        F.col("Type_Bilan").alias("type_bilan"),
        F.col("Fiscal_Year").cast("int").alias("fiscal_year"),
        F.col("Date_Cloture").cast("date").alias("date_cloture"),
        F.col("Total_Actif_EUR").cast("double").alias("total_actif_eur"),
        F.col("Actif_Immobilise_EUR").cast("double").alias("actif_immobilise_eur"),
        F.col("Actif_Circulant_EUR").cast("double").alias("actif_circulant_eur"),
        F.col("Capitaux_Propres_EUR").cast("double").alias("capitaux_propres_eur"),
        F.col("Provisions_Risques_Charges_EUR").cast("double").alias("provisions_risques_charges_eur"),
        F.col("Total_Dettes_EUR").cast("double").alias("total_dettes_eur"),
        F.col("Chiffre_Affaires_EUR").cast("double").alias("chiffre_affaires_eur"),
        F.col("Resultat_Net_EUR").cast("double").alias("resultat_net_eur"),
        F.col("Provisions_Techniques_Brutes_EUR").cast("double").alias("provisions_techniques_brutes_eur"),
        F.col("Bilan_Saisi_Id").alias("bilan_saisi_id"),
        F.lit(None).cast("string").alias("raw_liasse_json")
    )
    .withColumn(
        "net_margin",
        F.when(F.col("chiffre_affaires_eur") > 0,
               F.col("resultat_net_eur") / F.col("chiffre_affaires_eur"))
        .otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "leverage_ratio",
        F.when(F.col("capitaux_propres_eur") != 0,
        F.col("resultat_net_eur") / F.col("capitaux_propres_eur"))
        .otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "current_ratio",
        F.when(F.col("actif_circulant_eur") != 0,
        F.col("total_actif_eur") / F.col("actif_circulant_eur"))
        .otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "debt_to_equity_ratio",
        F.when(F.col("capitaux_propres_eur") != 0,
        F.col("total_dettes_eur") / F.col("capitaux_propres_eur"))
        .otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "debt_to_assets_ratio",
        F.when(F.col("total_actif_eur") != 0,
        F.col("total_dettes_eur") / F.col("total_actif_eur"))
        .otherwise(F.lit(None).cast("double")),
    )
    #red-flag boolean
    .withColumn("flag_capitaux_propres_negatifs", F.col("capitaux_propres_eur") < 0)
    .withColumn("flag_resultat_net_negatif", F.col("resultat_net_eur") <0)
    .withColumn(
        "flag_total_dettes_superieur_actif",
        F.col("total_dettes_eur") > F.col("total_actif_eur"),
    )
    .withColumn("silver_ingested_at", F.current_timestamp())
)
print(f"financial (pre-join) row count: {df_financial.count()}")



## Join `silver_companies` to compute `flag_capital_insuffisant`

In [0]:
df_companies_capital = spark.table(f"{SILVER_SCHEMA}.silver_company").select("siren", "capital_social_eur")

# display(df_companies_capital)

df_bilans = (
    df_bilans
    .join(df_companies_capital, on="siren", how="left")
    .withColumn(
        "flag_capital_insuffisant",
        F.when(
            F.col("capital_social_eur").isNotNull(),
            F.col("capitaux_propres_eur") < (F.col("capital_social_eur") / 2),
        ).otherwise(F.lit(None).cast("Boolean")),
    )
    .drop("capital_social_eur")
)

print(f"silver_bilans (final) rows count: {df_bilans.count()}")

In [0]:
(df_bilans.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", True)
 .saveAsTable(f"{SILVER_SCHEMA}.silver_bilans")
)
print(f"Saved {SILVER_SCHEMA}.silver_bilans")

In [0]:
#print SCHEMA
spark.table(f"{SILVER_SCHEMA}.silver_bilans").printSchema()


## Sanity Check

In [0]:
print("Bilans", spark.table(f"{SILVER_SCHEMA}.silver_bilans").count())
display(
    spark.table(f"{SILVER_SCHEMA}.silver_bilans")
    .select("siren", "fiscal_year", "capitaux_propres_eur","resultat_net_eur", "flag_capitaux_propres_negatifs", "flag_resultat_net_negatif", "flag_total_dettes_superieur_actif", "flag_capital_insuffisant")
)